<div style="display: flex; align-items: center;">

  <!-- Logos -->
  <div style="white-space: nowrap;">
    <img 
      src="https://www.upc.edu/comunicacio/ca/identitat/descarrega-arxius-grafics/fitxers-marca-principal/upc-positiu-p3005.png" 
      width="300"
      style="vertical-align: middle;"
    >
    <img 
      src="https://www.hipotecalowcost.com/wp-content/uploads/2019/08/Logo-CaixaBank.png" 
      width="200"
      style="vertical-align: middle;"
    >
  </div>

  <!-- Texto -->
  <div style="margin-left: auto; margin-right: 10px; text-align: right;">
      <p style="margin: 0;"><b>CaixaBank · Advanced Analytics Program</b></p>
      <p style="margin: 0;"><b>Model Risk & Data Science Training</b></p>
      <p style="margin: 0;">Automatización de procesos</p>
  </div>

</div>

# **LLM, prompting y agentes**
## *Introducción práctica a IA generativa aplicada a automatización*

En este laboratorio introduciremos modelos de lenguaje dentro del flujo de automatización.

La progresión será:

```text
texto
  ↓
prompt
  ↓
LLM
  ↓
respuesta
  ↓
validación
  ↓
decisión
  ↓
herramienta / acción
```

El objetivo no es tratar el LLM como una caja mágica, sino como un componente probabilístico que debe:

- recibir instrucciones claras;
- producir salidas estructuradas;
- validarse;
- estar limitado por reglas;
- escalar a revisión humana cuando sea necesario.

## Objetivos

Al finalizar deberías poder:

1. Explicar qué es un LLM.
2. Diferenciar generación, clasificación, extracción y resumen.
3. Diseñar prompts claros.
4. Separar instrucciones, contexto y datos.
5. Solicitar salidas estructuradas.
6. Validar JSON generado.
7. Detectar respuestas inválidas.
8. Implementar fallback.
9. Diseñar clasificación con confianza.
10. Derivar casos dudosos a revisión humana.
11. Entender qué es una herramienta.
12. Entender qué es un agente.
13. Diferenciar workflow determinista y agente.
14. Diseñar un agente limitado.
15. Preparar el paso hacia RAG.

## 1. Preparación

In [ ]:
from pathlib import Path
import json
import re
import random

import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)

RAIZ = Path(
    "demo_llm_agentes"
)

RAIZ.mkdir(
    exist_ok=True
)

print(
    "Entorno preparado."
)

# 2. ¿Qué es un LLM?

Un Large Language Model procesa y genera lenguaje a partir de patrones aprendidos.

En automatización puede ayudarnos con tareas donde las reglas clásicas empiezan a ser insuficientes:

- clasificar texto libre;
- extraer entidades;
- resumir;
- reescribir;
- decidir qué herramienta usar;
- interpretar documentos.

Pero sus respuestas no son garantías matemáticas.

Por eso necesitamos validación.

## 3. Tareas típicas

In [ ]:
tareas_llm = pd.DataFrame({
    "tarea": [
        "Clasificación",
        "Extracción",
        "Resumen",
        "Generación",
        "Reescritura",
        "Decisión"
    ],
    "ejemplo": [
        "tipo de incidencia",
        "nombre/fecha/importe",
        "resumen ejecutivo",
        "respuesta inicial",
        "tono formal",
        "qué herramienta utilizar"
    ]
})

tareas_llm

# 4. Prompt básico

In [ ]:
prompt_basico = '''
Clasifica el siguiente mensaje como:
- venta
- soporte
- otro

Mensaje:
"Necesito cambiar la contraseña de mi cuenta"
'''

print(
    prompt_basico
)

### Problema

El prompt funciona, pero puede ser demasiado abierto.

No hemos especificado:

- formato de salida;
- qué hacer si hay ambigüedad;
- si debe justificar;
- si puede inventar categorías.

# 5. Prompt estructurado

In [ ]:
prompt_mejor = '''
Eres un clasificador de solicitudes.

Categorías permitidas:
- venta
- soporte
- otro

Reglas:
1. Devuelve únicamente una categoría válida.
2. No inventes nuevas categorías.
3. Si el mensaje es ambiguo, devuelve "otro".

Mensaje:
"Necesito cambiar la contraseña de mi cuenta"
'''

print(
    prompt_mejor
)

### Interpretación

Un prompt mejor reduce grados de libertad.

La idea no es “hacerlo más largo”, sino **hacer explícito el contrato**.

# 6. Estructura recomendada de prompt

In [ ]:
estructura_prompt = pd.DataFrame({
    "bloque": [
        "Rol",
        "Objetivo",
        "Contexto",
        "Reglas",
        "Entrada",
        "Formato"
    ],
    "funcion": [
        "Qué papel adopta",
        "Qué debe conseguir",
        "Información necesaria",
        "Límites",
        "Datos concretos",
        "Cómo responder"
    ]
})

estructura_prompt

# 7. Clasificación con salida JSON

In [ ]:
prompt_json = '''
Clasifica el mensaje.

Categorías:
- venta
- soporte
- otro

Devuelve JSON con:
{
  "categoria": "...",
  "confianza": 0.0,
  "razon": "..."
}

Mensaje:
"Quiero saber el precio del plan profesional"
'''

print(
    prompt_json
)

# 8. Simular respuesta LLM

In [ ]:
respuesta_llm = '''
{
  "categoria": "venta",
  "confianza": 0.93,
  "razon": "El usuario pregunta por el precio de un plan."
}
'''

print(
    respuesta_llm
)

## 8.1 Parsear JSON

In [ ]:
datos_respuesta = json.loads(
    respuesta_llm
)

datos_respuesta

# 9. Validar salida estructurada

In [ ]:
CATEGORIAS = {
    "venta",
    "soporte",
    "otro"
}


def validar_salida_clasificacion(
    respuesta
):
    errores = []

    if not isinstance(
        respuesta,
        dict
    ):
        return [
            "La respuesta no es un objeto."
        ]

    for campo in [
        "categoria",
        "confianza",
        "razon"
    ]:
        if campo not in respuesta:
            errores.append(
                f"Falta {campo}"
            )

    if (
        "categoria"
        in respuesta
        and respuesta[
            "categoria"
        ]
        not in CATEGORIAS
    ):
        errores.append(
            "Categoría inválida."
        )

    if (
        "confianza"
        in respuesta
    ):
        confianza = respuesta[
            "confianza"
        ]

        if (
            not isinstance(
                confianza,
                (
                    int,
                    float
                )
            )
            or confianza < 0
            or confianza > 1
        ):
            errores.append(
                "Confianza fuera de rango."
            )

    return errores


validar_salida_clasificacion(
    datos_respuesta
)

# 10. Provocar salida inválida

In [ ]:
respuesta_invalida = {
    "categoria":
        "facturacion",
    "confianza":
        1.7
}

validar_salida_clasificacion(
    respuesta_invalida
)

### Interpretación

Aunque pidamos JSON, debemos validar:

- estructura;
- categorías;
- rangos;
- campos obligatorios.

# 11. Revisión humana por confianza

In [ ]:
def decidir_revision(
    respuesta,
    umbral=0.80
):
    errores = validar_salida_clasificacion(
        respuesta
    )

    if errores:
        return {
            "estado":
                "revision",
            "motivo":
                errores
        }

    if respuesta[
        "confianza"
    ] < umbral:
        return {
            "estado":
                "revision",
            "motivo":
                [
                    "Confianza insuficiente."
                ]
        }

    return {
        "estado":
            "automatico",
        "motivo":
            []
    }


decidir_revision(
    datos_respuesta
)

# 12. Simular lote de clasificaciones

In [ ]:
mensajes = pd.DataFrame({
    "mensaje": [
        "Quiero contratar el servicio premium",
        "No puedo iniciar sesión",
        "Gracias por vuestra ayuda",
        "Necesito cambiar mi tarjeta",
        "¿Tenéis descuentos?"
    ],
    "categoria_simulada": [
        "venta",
        "soporte",
        "otro",
        "soporte",
        "venta"
    ],
    "confianza_simulada": [
        0.95,
        0.91,
        0.54,
        0.76,
        0.88
    ]
})

mensajes

## 12.1 Aplicar revisión

In [ ]:
mensajes[
    "estado"
] = mensajes.apply(
    lambda fila:
        (
            "automatico"
            if fila[
                "confianza_simulada"
            ] >= 0.80
            else "revision"
        ),
    axis=1
)

mensajes

## 12.2 Visualizar

In [ ]:
conteo = (
    mensajes[
        "estado"
    ]
    .value_counts()
)

plt.figure(
    figsize=(7, 5)
)

plt.bar(
    conteo.index,
    conteo.values
)

plt.ylabel(
    "Mensajes"
)

plt.title(
    "Automático vs revisión humana"
)

plt.tight_layout()
plt.show()

# 13. Extracción estructurada

In [ ]:
texto_factura = '''
Factura F-2026-0041
Cliente: Empresa Delta
Fecha: 2026-08-15
Total: 1.245,80 EUR
'''

prompt_extraccion = f'''
Extrae del texto:
- numero_factura
- cliente
- fecha
- total

Devuelve JSON.

Texto:
{texto_factura}
'''

print(
    prompt_extraccion
)

# 13.1 Simular extracción

In [ ]:
extraccion = {
    "numero_factura":
        "F-2026-0041",
    "cliente":
        "Empresa Delta",
    "fecha":
        "2026-08-15",
    "total":
        1245.80
}

extraccion

# 13.2 Validar extracción

In [ ]:
def validar_factura(
    datos
):
    errores = []

    requeridos = {
        "numero_factura",
        "cliente",
        "fecha",
        "total"
    }

    faltantes = (
        requeridos
        - set(
            datos.keys()
        )
    )

    if faltantes:
        errores.append(
            f"Faltan campos: {sorted(faltantes)}"
        )

    if (
        "total"
        in datos
        and (
            not isinstance(
                datos[
                    "total"
                ],
                (
                    int,
                    float
                )
            )
            or datos[
                "total"
            ] < 0
        )
    ):
        errores.append(
            "Total inválido."
        )

    return errores


validar_factura(
    extraccion
)

# 14. Resumen

In [ ]:
texto_largo = '''
La empresa ha incrementado las ventas del canal online,
pero presenta un aumento moderado de incidencias en soporte.
El área de operaciones mantiene tiempos estables.
El principal riesgo identificado está en la concentración
de ventas en dos clientes principales.
'''

prompt_resumen = f'''
Resume el texto en tres puntos breves.
No añadas información externa.

Texto:
{texto_largo}
'''

print(
    prompt_resumen
)

### Regla importante

Cuando utilizamos un LLM sobre contenido específico, debemos indicar si puede o no incorporar conocimiento externo.

# 15. Hallucination / invención de información

Un LLM puede producir una respuesta plausible pero no sustentada por la entrada.

Ejemplo problemático:

```text
Texto: "Las ventas han aumentado."

Respuesta inventada:
"Las ventas aumentaron un 23% gracias a la campaña de verano."
```

El porcentaje y la causa no estaban presentes.

En automatización esto puede convertirse en un error operativo.

# 16. Prompt con restricción de evidencia

In [ ]:
prompt_evidencia = '''
Responde exclusivamente con información presente en el texto.
Si un dato no aparece, utiliza null.

Devuelve JSON con:
- crecimiento_porcentaje
- causa
- tendencia

Texto:
"Las ventas han aumentado."
'''

print(
    prompt_evidencia
)

# 17. Few-shot prompting

Podemos incluir ejemplos para enseñar el formato.

```text
Entrada: "No puedo acceder a mi cuenta"
Salida: soporte

Entrada: "¿Cuánto cuesta el plan?"
Salida: venta

Entrada: "Gracias"
Salida: otro
```

Esto puede mejorar consistencia cuando la tarea tiene patrones claros.

# 18. Zero-shot vs few-shot

In [ ]:
comparativa_prompting = pd.DataFrame({
    "tipo": [
        "Zero-shot",
        "Few-shot"
    ],
    "descripcion": [
        "Instrucción sin ejemplos",
        "Instrucción con ejemplos"
    ],
    "uso": [
        "Tarea clara",
        "Formato o criterio complejo"
    ]
})

comparativa_prompting

# 19. Temperatura: concepto

La temperatura controla conceptualmente cuánto varía la generación.

Para tareas de automatización estructurada solemos preferir baja variabilidad.

Ejemplos:

```text
clasificar factura → baja variabilidad
escribir campaña creativa → mayor variabilidad
```

El parámetro concreto depende del proveedor/modelo.

# 20. Determinismo vs probabilidad

In [ ]:
comparativa = pd.DataFrame({
    "tipo": [
        "Regla Python",
        "LLM"
    ],
    "misma_entrada": [
        "Misma salida esperada",
        "Puede variar"
    ],
    "ideal_para": [
        "Reglas conocidas",
        "Lenguaje ambiguo"
    ],
    "validacion": [
        "Tests",
        "Tests + esquema + revisión"
    ]
})

comparativa

# 21. Workflow híbrido reglas + LLM

Arquitectura recomendable:

```text
entrada
   ↓
¿regla determinista?
 ├─ sí → Python / n8n
 └─ no → LLM
           ↓
      validar JSON
           ↓
      confianza suficiente?
       ├─ sí → acción
       └─ no → revisión
```

# 22. Herramientas

Una **herramienta** es una capacidad externa que el modelo puede utilizar.

Ejemplos:

- consultar una API;
- buscar un cliente;
- generar un informe;
- guardar un archivo;
- ejecutar una función;
- consultar una base documental.

El modelo no debería ejecutar arbitrariamente cualquier cosa.

Las herramientas deben estar delimitadas.

# 22.1 Definir herramientas conceptuales

In [ ]:
herramientas = pd.DataFrame({
    "nombre": [
        "buscar_cliente",
        "consultar_ventas",
        "crear_informe",
        "enviar_notificacion"
    ],
    "entrada": [
        "cliente_id",
        "cliente_id",
        "cliente_id, periodo",
        "email, mensaje"
    ],
    "salida": [
        "datos cliente",
        "ventas",
        "ruta informe",
        "estado"
    ]
})

herramientas

# 23. ¿Qué es un agente?

Un agente combina:

- modelo;
- instrucciones;
- herramientas;
- memoria/contexto;
- lógica de decisión.

Conceptualmente:

```text
objetivo
   ↓
LLM
   ↓
¿necesito herramienta?
   ├─ no → responder
   └─ sí → ejecutar herramienta
              ↓
           observar resultado
              ↓
           decidir siguiente paso
```

No significa dar acceso ilimitado al sistema.

# 24. Workflow vs agente

In [ ]:
workflow_vs_agente = pd.DataFrame({
    "criterio": [
        "Ruta",
        "Control",
        "Flexibilidad",
        "Predicción",
        "Uso"
    ],
    "Workflow": [
        "Predefinida",
        "Alto",
        "Media",
        "Alta",
        "Procesos conocidos"
    ],
    "Agente": [
        "Decidida parcialmente",
        "Debe limitarse",
        "Alta",
        "Menor",
        "Problemas abiertos"
    ]
})

workflow_vs_agente

# 25. Diseñar un agente limitado

In [ ]:
agente = {
    "objetivo":
        "Responder consultas de clientes",

    "herramientas_permitidas": [
        "buscar_cliente",
        "consultar_ventas"
    ],

    "acciones_prohibidas": [
        "eliminar cliente",
        "modificar facturación",
        "enviar pagos"
    ],

    "requiere_revision": [
        "respuesta con confianza baja",
        "solicitud fuera de alcance"
    ]
}

agente

### Interpretación

Un agente profesional necesita límites explícitos.

No debemos confundir autonomía con ausencia de control.

# 26. Simular selección de herramienta

In [ ]:
consultas = pd.DataFrame({
    "consulta": [
        "¿Cuál es el email del cliente C001?",
        "¿Cuánto vendió C002?",
        "Hola, gracias"
    ],
    "herramienta_simulada": [
        "buscar_cliente",
        "consultar_ventas",
        None
    ]
})

consultas

# 27. Tool calling conceptual

In [ ]:
llamada_herramienta = {
    "tool":
        "consultar_ventas",

    "arguments": {
        "cliente_id":
            "C002"
    }
}

print(
    json.dumps(
        llamada_herramienta,
        indent=2,
        ensure_ascii=False
    )
)

# 28. Validar llamada a herramienta

In [ ]:
HERRAMIENTAS_PERMITIDAS = {
    "buscar_cliente",
    "consultar_ventas",
    "crear_informe"
}


def validar_tool_call(
    llamada
):
    errores = []

    if llamada.get(
        "tool"
    ) not in HERRAMIENTAS_PERMITIDAS:
        errores.append(
            "Herramienta no permitida."
        )

    if not isinstance(
        llamada.get(
            "arguments"
        ),
        dict
    ):
        errores.append(
            "arguments debe ser un objeto."
        )

    return errores


validar_tool_call(
    llamada_herramienta
)

# 29. Provocar herramienta prohibida

In [ ]:
llamada_peligrosa = {
    "tool":
        "eliminar_cliente",

    "arguments": {
        "cliente_id":
            "C002"
    }
}

validar_tool_call(
    llamada_peligrosa
)

### Interpretación

La salida del modelo no debe convertirse automáticamente en una acción.

Primero validamos que la herramienta esté permitida y que los argumentos sean correctos.

# 30. Agente con human-in-the-loop

Patrón:

```text
agente propone acción
      ↓
¿acción sensible?
  ├─ no → ejecutar
  └─ sí → solicitar aprobación
```

Ejemplos sensibles:

- enviar comunicación externa;
- modificar registros;
- borrar;
- aprobar gasto;
- ejecutar acciones irreversibles.

# 31. Estado de un agente

In [ ]:
estado_agente = {
    "objetivo":
        "Responder consulta",

    "pasos": [
        {
            "tipo":
                "tool",
            "nombre":
                "buscar_cliente",
            "resultado":
                "Cliente encontrado"
        },
        {
            "tipo":
                "tool",
            "nombre":
                "consultar_ventas",
            "resultado":
                125000
        }
    ],

    "estado":
        "completado"
}

estado_agente

# 32. Memoria: concepto

Un agente puede necesitar contexto de una conversación o proceso.

Pero memoria no significa guardar todo indiscriminadamente.

Tipos conceptuales:

- contexto de ejecución;
- historial conversacional;
- estado del workflow;
- memoria externa persistente.

En el siguiente laboratorio veremos otra forma de contexto: **RAG**.

# 33. Diseñar prompt de agente

In [ ]:
prompt_agente = '''
Eres un asistente de gestión.

Objetivo:
Responder consultas sobre clientes y ventas.

Herramientas disponibles:
- buscar_cliente(cliente_id)
- consultar_ventas(cliente_id)

Reglas:
1. No inventes datos.
2. Si necesitas información, usa una herramienta.
3. No ejecutes acciones fuera de las herramientas permitidas.
4. Si la solicitud queda fuera de alcance, indica que requiere revisión.
'''

print(
    prompt_agente
)

# 34. LLM dentro de n8n

En n8n el patrón puede ser:

```text
Webhook
   ↓
AI Agent / LLM node
   ↓
herramientas
   ├─ HTTP Request
   ├─ workflow
   ├─ base de datos
   └─ código
   ↓
respuesta
```

Las integraciones concretas pueden variar según versión y proveedor, pero el modelo mental se mantiene.

# 35. Preparar dataset para práctica

In [ ]:
solicitudes = pd.DataFrame({
    "id": [
        1,
        2,
        3,
        4,
        5,
        6
    ],
    "texto": [
        "Quiero contratar el plan premium",
        "No puedo entrar en mi cuenta",
        "Gracias por la ayuda",
        "Quiero cambiar mi método de pago",
        "¿Cuánto cuesta la versión empresa?",
        "Necesito hablar con alguien"
    ]
})

solicitudes

# 36. Clasificación simulada para validar pipeline

In [ ]:
def clasificador_simulado(
    texto
):
    texto_lower = (
        texto.lower()
    )

    if any(
        palabra
        in texto_lower
        for palabra
        in [
            "contratar",
            "cuánto cuesta",
            "precio"
        ]
    ):
        return {
            "categoria":
                "venta",
            "confianza":
                0.92
        }

    if any(
        palabra
        in texto_lower
        for palabra
        in [
            "entrar",
            "pago",
            "método"
        ]
    ):
        return {
            "categoria":
                "soporte",
            "confianza":
                0.88
        }

    return {
        "categoria":
            "otro",
        "confianza":
            0.60
    }


resultados = []

for _, fila in solicitudes.iterrows():

    pred = clasificador_simulado(
        fila[
            "texto"
        ]
    )

    resultados.append({
        "id":
            fila[
                "id"
            ],

        "texto":
            fila[
                "texto"
            ],

        "categoria":
            pred[
                "categoria"
            ],

        "confianza":
            pred[
                "confianza"
            ],

        "estado":
            (
                "automatico"
                if pred[
                    "confianza"
                ] >= 0.80
                else "revision"
            )
    })


df_resultados = pd.DataFrame(
    resultados
)

df_resultados

# 36.1 Visualizar categorías

In [ ]:
categoria = (
    df_resultados[
        "categoria"
    ]
    .value_counts()
)

plt.figure(
    figsize=(7, 5)
)

plt.bar(
    categoria.index,
    categoria.values
)

plt.ylabel(
    "Solicitudes"
)

plt.title(
    "Clasificación de solicitudes"
)

plt.tight_layout()
plt.show()

# 37. Especificación de salida LLM

In [ ]:
schema_salida = {
    "categoria":
        "venta | soporte | otro",
    "confianza":
        "float 0..1",
    "razon":
        "string breve"
}

ruta_schema = (
    RAIZ
    / "schema_clasificacion.json"
)

ruta_schema.write_text(
    json.dumps(
        schema_salida,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print(
    ruta_schema.read_text(
        encoding="utf-8"
    )
)

# 38. Checklist de uso de LLM

- [ ] ¿La tarea necesita realmente LLM?
- [ ] ¿Existe una regla determinista mejor?
- [ ] ¿El prompt tiene objetivo claro?
- [ ] ¿Las categorías están cerradas?
- [ ] ¿La salida está estructurada?
- [ ] ¿Valido JSON?
- [ ] ¿Valido rangos?
- [ ] ¿Existe umbral de confianza?
- [ ] ¿Hay revisión humana?
- [ ] ¿Evito datos inventados?
- [ ] ¿Las herramientas están limitadas?
- [ ] ¿Las acciones sensibles requieren aprobación?
- [ ] ¿Registro la decisión?

# 39. Mini-práctica integrada

Diseña un clasificador inteligente de solicitudes.

Debe:

1. recibir texto;
2. construir prompt;
3. exigir JSON;
4. devolver categoría;
5. devolver confianza;
6. validar esquema;
7. rechazar categoría desconocida;
8. enviar confianza baja a revisión;
9. registrar decisión;
10. seleccionar herramienta cuando sea necesario.

Después intégralo conceptualmente en n8n:

```text
Webhook
   ↓
LLM
   ↓
validación
   ↓
IF confianza
 ┌───────┴────────┐
 ↓                ↓
acción          revisión
```

# 40. Ejercicios finales

### Ejercicio 1
Diseña un prompt de clasificación.

### Ejercicio 2
Añade categorías cerradas.

### Ejercicio 3
Solicita JSON.

### Ejercicio 4
Parsea respuesta.

### Ejercicio 5
Valida campos.

### Ejercicio 6
Valida confianza.

### Ejercicio 7
Provoca categoría inválida.

### Ejercicio 8
Añade revisión humana.

### Ejercicio 9
Diseña extracción de factura.

### Ejercicio 10
Valida total.

### Ejercicio 11
Crea prompt de resumen sin información externa.

### Ejercicio 12
Diseña few-shot.

### Ejercicio 13
Compara regla y LLM.

### Ejercicio 14
Define tres herramientas.

### Ejercicio 15
Valida tool calls.

### Ejercicio 16
Bloquea una herramienta peligrosa.

### Ejercicio 17
Diseña un agente limitado.

### Ejercicio 18
Reto: integra clasificación + herramientas + revisión humana.

# 41. Preguntas de reflexión

1. ¿Qué es un LLM?
2. ¿Qué diferencia hay entre clasificación y generación?
3. ¿Por qué estructurar prompts?
4. ¿Por qué pedir JSON?
5. ¿Por qué validar incluso si el modelo sigue instrucciones?
6. ¿Qué es una alucinación?
7. ¿Cuándo usar reglas?
8. ¿Cuándo usar LLM?
9. ¿Qué es confianza?
10. ¿Qué papel tiene revisión humana?
11. ¿Qué es una herramienta?
12. ¿Qué es un agente?
13. ¿Qué diferencia hay entre workflow y agente?
14. ¿Por qué limitar herramientas?
15. ¿Cómo prepara esto el siguiente laboratorio de RAG?

# 42. Cierre

Ya tenemos:

```text
LLM
 ↓
prompt
 ↓
salida estructurada
 ↓
validación
 ↓
herramientas
 ↓
agente
```

El siguiente laboratorio añadirá conocimiento externo controlado:

```text
documentos
   ↓
chunking
   ↓
embeddings
   ↓
búsqueda semántica
   ↓
RAG
   ↓
respuesta sustentada
```